# 第20课：RAG 调优与结构化溯源

本笔记本是课堂讲义。每个知识点包含：理论知识、案例代码、讲解、易错点与练习。综合练习 P1 使用课程根目录的教学运行时 [`agent_lab`](../agent_lab/README.md)。课后独立练习见 [chapter20_RAG调优与结构化溯源_课后练习.ipynb](chapter20_RAG调优与结构化溯源_课后练习.ipynb)。

**阶段定位**：阶段四 · 任务三 3.2 / 25分。25分

检索能用之后要**准**、要**可追溯**、要**会拒绝**。任务三 3.2 共 25 分：单文档 7、跨文档 9、越界兜底 9。

## 学习目标

1. 说明混合检索（余弦 + BM25）与截断重排。
2. 用 SourcedAnswer 强制携带 chunk_id、置信度、原文片段。
3. 对知识库外问题返回 FallbackResponse，而不是幻觉。

## 学习知识点

| 基础 7分 | 进阶 9分 | 高难 9分 |
| --- | --- | --- |
| 单文档事实 | 跨文档对比 | 越界 / 幻觉对抗 |
| 带 sources | 至少两个 doc_id | kind=fallback |

## 基础回顾与案例提问

1. **R.1** 只向量或只 BM25，短中文查询可能怎样？
2. **R.2** 回答很对但没有 chunk_id，本课算不算合格溯源？
3. **R.3** “明天深圳天气怎么样？”应走哪一种实体？

OUT_OF_CORPUS 关键词含天气、股价、中奖等。不要把兜底做成“我猜可能下雨”。

使用 Python 3；需要 `pydantic`。从本课文件夹启动内核。本课不要求 GPU，也不强制安装 `langgraph` / `openai`。未配置私有化端点时，`get_client()` 返回进程内 Fake。不要使用 pandas。综合练习不要抄 `experiment.py` 的整段答案，按题面逐步完成。


In [ ]:
# R.1–R.3: Write and verify your predictions here.


In [ ]:
import sys
from pathlib import Path

COURSE = Path.cwd().resolve()
if COURSE.name.startswith("第"):
    COURSE = COURSE.parent
if str(COURSE) not in sys.path:
    sys.path.insert(0, str(COURSE))
print("已加入路径:", COURSE)
print("请从本课文件夹启动内核。未配置 OPENAI_BASE_URL 时使用教学 Fake 端点，不要求 GPU。")


## 1. 混合检索

### 理论知识

**余弦捕捉重叠词，BM25 惩罚过长块、奖励罕见词。** 本课 0.6 * cosine + 0.4 * bm25。

### 案例：打开检索


In [ ]:
from pathlib import Path
from agent_lab.rag import Retriever, load_corpus, answer_with_rag
KB = Path.cwd().parent / "agent_lab" / "knowledge"
retriever = Retriever(load_corpus(KB))
for h in retriever.search("对比价格与续航", k=3, hybrid=True):
    print(h.score, h.chunk_id)


### 讲解

hybrid=False 可对照。作业要能说出一句话差异，不要求调参比赛。

### 易错点与练习

1. **K1.1** b 参数与文档长度有何关系（直觉即可）？
2. **K1.2** 为什么重排先取 2k 再截回 k？本课是教学截断。

**作答：** ____。


In [ ]:
# Write your predictions, reasoning, or solution here.
# Keep deliberately faulty snippets in Markdown; run your corrected code here.


## 2. 结构化溯源

### 理论知识

**SourcedAnswer.sources 每项含 chunk_id、confidence、quote。**

### 案例：看模型


In [ ]:
from agent_lab.rag import SourcedAnswer, FallbackResponse
print(SourcedAnswer.model_json_schema()["properties"].keys())
print(FallbackResponse.model_json_schema()["properties"].keys())


### 讲解

两种实体用 kind 区分：answer 或 fallback。不要混成一个随便 dict。

### 易错点与练习

1. **K2.1** confidence 在教学里等于检索 score，工业上还可能来自 Rerank 模型。差别？
2. **K2.2** quote 截到 80 字是为了什么？

**作答：** ____。


In [ ]:
# Write your predictions, reasoning, or solution here.
# Keep deliberately faulty snippets in Markdown; run your corrected code here.


## 3. 基础：单文档精准定位

### 理论知识

**事实型：Widget-X 续航多久？** 应引用手册切片。

### 案例：基础


In [ ]:
basic = answer_with_rag(retriever, "Widget-X 续航多久？")
print(basic["kind"], basic.get("sources", [{}])[0].get("chunk_id"))
print(basic["answer"][:80])


### 讲解

7 分看格式规整 + 来源切片，不看文采。

### 易错点与练习

1. **K3.1** 若 sources 为空但 answer 很长，如何判？
2. **K3.2** chunk_id 应以 manual 为主还是 price？

**作答：** ____。


In [ ]:
# Write your predictions, reasoning, or solution here.
# Keep deliberately faulty snippets in Markdown; run your corrected code here.


## 4. 进阶：跨文档聚合

### 理论知识

**价格在 price.md，续航在 manual.md。** 对比题必须两边都出现在 sources 的 doc_id 集合里。

### 案例：对比


In [ ]:
advanced = answer_with_rag(retriever, "对比 Widget-X 和 Widget-Mini 的价格与续航")
print([s["chunk_id"] for s in advanced.get("sources", [])])
print({s["chunk_id"].split("::")[0] for s in advanced["sources"]})


### 讲解

可用 Markdown 表在作业里手工整理对比，但数据必须来自 hits，禁止凭记忆填 1299。

### 易错点与练习

1. **K4.1** 不相连片段指的是什么？
2. **K4.2** 只有 price::0 三条重复算不算跨文档？

**作答：** ____。


In [ ]:
# Write your predictions, reasoning, or solution here.
# Keep deliberately faulty snippets in Markdown; run your corrected code here.


## 5. 高难：越界提问

### 理论知识

**知识库外必须拒绝编造。** 天气触发 out_of_corpus。

### 案例：天气


In [ ]:
hard = answer_with_rag(retriever, "明天深圳天气怎么样？")
print(hard)
assert hard["kind"] == "fallback" 


### 讲解

FallbackResponse 含 reason、message、question。这是防御，不是答非所问。

### 易错点与练习

1. **K5.1** 若去掉关键词列表、只靠 score<0.05，天气题还稳吗？对照教学实现。
2. **K5.2** 把天气编成“小雨转晴”会在哪一档零分？

**作答：** ____。


In [ ]:
# Write your predictions, reasoning, or solution here.
# Keep deliberately faulty snippets in Markdown; run your corrected code here.


## 6. 低置信兜底

### 理论知识

**库内但分数过低也会 fallback。** reason=low_confidence。

### 案例：读阈值


In [ ]:
print("hits[0].score < 0.05 -> low_confidence")


### 讲解

阈值是教学常数。作业解释它的存在即可，不必网格搜索。

### 易错点与练习

1. **K6.1** 阈过高会怎样？过低会怎样？
2. **K6.2** 与越界关键词防御如何叠加？

**作答：** ____。


In [ ]:
# Write your predictions, reasoning, or solution here.
# Keep deliberately faulty snippets in Markdown; run your corrected code here.


## 7. 幻觉对抗清单

### 理论知识

**对抗不是攻击学校，是验收。** 准备：库内事实、跨库对比、库外诱导。

### 案例：三类问题


In [ ]:
probes = ["Widget-X 续航多久？", "对比 Widget-X 和 Widget-Mini 的价格与续航", "明天深圳天气怎么样？"]
for q in probes:
    a = answer_with_rag(retriever, q)
    print(q, "->", a["kind"], a.get("reason"))


### 讲解

学期收束：Serving 约束输出、图约束流程、Schema 约束字段、RAG 约束证据。

### 易错点与练习

1. **K7.1** 哪一层防的是“说得像真的”？
2. **K7.2** 哪一层防的是“无限循环”？

**作答：** ____。


In [ ]:
# Write your predictions, reasoning, or solution here.
# Keep deliberately faulty snippets in Markdown; run your corrected code here.


## 8. 25 分评分对照

### 理论知识

**7+9+9。** experiment.py 三连断言。

### 案例：自检


In [ ]:
assert basic["kind"] == "answer" and basic["sources"]
assert len({s["chunk_id"].split("::")[0] for s in advanced["sources"]}) >= 2
assert hard["reason"] == "out_of_corpus"
print("3.2 课堂自检通过")


### 讲解

完成后运行本课 experiment.py。不要改知识库来让天气变成手册内容。

### 易错点与练习

1. **K8.1** 跨文档不足两个 doc_id 扣哪 9 分？
2. **K8.2** fallback 的 message 需要文学性吗？

**作答：** ____。


In [ ]:
# Write your predictions, reasoning, or solution here.
# Keep deliberately faulty snippets in Markdown; run your corrected code here.


## 综合练习：溯源与兜底

按 P1.1 → P1.2 → P1.3 顺序完成。每步都要能独立看出你做了什么。


### P1.1　单文档

提问续航，打印 answer 与 sources。


In [ ]:
# P1.1: single-doc sourced answer.


### P1.2　跨文档

对比价格与续航，列出 doc_id 集合。


In [ ]:
# P1.2: multi-doc sources.


### P1.3　越界

天气问题必须 kind=fallback。


In [ ]:
# P1.3: fallback.


课后请打开 [chapter20_RAG调优与结构化溯源_课后练习.ipynb](chapter20_RAG调优与结构化溯源_课后练习.ipynb)。P1 基础、P2 进阶对比表、P3 选做再写一个越界问题。本模块到此收束。
